In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, sum as spark_sum

spark = SparkSession.builder.appName("inventory_management").getOrCreate()

In [0]:
from pyspark.sql.functions import *

In [0]:
products_data = [
    (1, "Laptop", 10),
    (2, "Mouse", 20),
    (3, "Keyboard", 15),
    (4, "Monitor", 8)
]

products_df = spark.createDataFrame(products_data,
    ["product_id", "product_name", "reorder_level"])

warehouses_data = [
    (1, "Main Warehouse", "Chennai"),
    (2, "Backup Warehouse", "Bangalore")
]

warehouses_df = spark.createDataFrame(warehouses_data,
    ["warehouse_id", "warehouse_name", "location"])

stock_data = [
    (1, 1, 1, 50, "IN", "2026-01-10"),
    (2, 2, 1, 30, "IN", "2026-01-12"),
    (3, 3, 2, 20, "IN", "2026-01-15"),
    (4, 1, 1, 10, "OUT", "2026-01-18"),
    (5, 2, 1, None, "OUT", "2026-01-20"),
    (6, 3, 2, -5, "IN", "2026-01-22"),
    (7, 4, 2, 15, "IN", "invalid_date") 
]

stock_df = spark.createDataFrame(stock_data,
    ["movement_id", "product_id", "warehouse_id", "quantity", "movement_type", "movement_date"])

In [0]:
products_df.show()
warehouses_df.show()
stock_df.show()

+----------+------------+-------------+
|product_id|product_name|reorder_level|
+----------+------------+-------------+
|         1|      Laptop|           10|
|         2|       Mouse|           20|
|         3|    Keyboard|           15|
|         4|     Monitor|            8|
+----------+------------+-------------+

+------------+----------------+---------+
|warehouse_id|  warehouse_name| location|
+------------+----------------+---------+
|           1|  Main Warehouse|  Chennai|
|           2|Backup Warehouse|Bangalore|
+------------+----------------+---------+

+-----------+----------+------------+--------+-------------+-------------+------------+
|movement_id|product_id|warehouse_id|quantity|movement_type|movement_date|adjusted_qty|
+-----------+----------+------------+--------+-------------+-------------+------------+
|          1|         1|           1|      50|           IN|   2026-01-10|          50|
|          2|         2|           1|      30|           IN|   2026-01-12|

In [0]:
stock_df = stock_df.fillna({"quantity": 0})

stock_df = stock_df.withColumn(
    "quantity",
    when(col("quantity") < 0, 0).otherwise(col("quantity"))
)

stock_df = stock_df.withColumn(
    "movement_date",
    expr("try_to_date(movement_date, 'yyyy-MM-dd')")
)

stock_df = stock_df.filter(col("movement_date").isNotNull())

In [0]:
stock_df = stock_df.withColumn(
    "adjusted_qty",
    when(col("movement_type") == "OUT", -col("quantity"))
    .otherwise(col("quantity"))
)

In [0]:
product_stock_df = stock_df.groupBy("product_id").agg(
    spark_sum("adjusted_qty").alias("current_stock")
)

display(product_stock_df)

product_id,current_stock
1,40
2,30
3,20


In [0]:
warehouse_stock_df = stock_df.groupBy("warehouse_id").agg(
    spark_sum("adjusted_qty").alias("total_stock")
)

display(warehouse_stock_df)

warehouse_id,total_stock
1,70
2,20


In [0]:
inventory_df = stock_df.join(products_df, "product_id") \
    .join(warehouses_df, "warehouse_id")

display(inventory_df)

warehouse_id,product_id,movement_id,quantity,movement_type,movement_date,adjusted_qty,product_name,reorder_level,warehouse_name,location
1,1,1,50,IN,2026-01-10,50,Laptop,10,Main Warehouse,Chennai
1,2,2,30,IN,2026-01-12,30,Mouse,20,Main Warehouse,Chennai
2,3,3,20,IN,2026-01-15,20,Keyboard,15,Backup Warehouse,Bangalore
1,1,4,10,OUT,2026-01-18,-10,Laptop,10,Main Warehouse,Chennai
1,2,5,0,OUT,2026-01-20,0,Mouse,20,Main Warehouse,Chennai
2,3,6,0,IN,2026-01-22,0,Keyboard,15,Backup Warehouse,Bangalore


In [0]:
final_stock_df = product_stock_df.join(products_df, "product_id")

low_stock_df = final_stock_df.withColumn(
    "status",
    when(col("current_stock") < col("reorder_level"), "reorder")
    .otherwise("sufficient")
)

display(low_stock_df)

product_id,current_stock,product_name,reorder_level,status
1,40,Laptop,10,sufficient
2,30,Mouse,20,sufficient
3,20,Keyboard,15,sufficient


In [0]:
warehouse_status_df = warehouse_stock_df.withColumn(
    "warehouse_status",
    when(col("total_stock") < 50, "understock")
    .when(col("total_stock") > 100, "overstock")
    .otherwise("balanced")
)

display(warehouse_status_df)

warehouse_id,total_stock,warehouse_status
1,70,balanced
2,20,understock


In [0]:
master_df = inventory_df.groupBy(
    "product_id", "product_name", "warehouse_name", "reorder_level"
).agg(
    spark_sum("adjusted_qty").alias("current_stock")
)

master_df = master_df.withColumn(
    "reorder_flag",
    when(col("current_stock") < col("reorder_level"), "YES")
    .otherwise("NO")
)

display(master_df)

product_id,product_name,warehouse_name,reorder_level,current_stock,reorder_flag
1,Laptop,Main Warehouse,10,40,NO
2,Mouse,Main Warehouse,20,30,NO
3,Keyboard,Backup Warehouse,15,20,NO


In [0]:
master_df.write.format("delta").mode("overwrite").saveAsTable("inventory")

In [0]:
spark.sql("select * from inventory").show()

+----------+------------+----------------+-------------+-------------+------------+
|product_id|product_name|  warehouse_name|reorder_level|current_stock|reorder_flag|
+----------+------------+----------------+-------------+-------------+------------+
|         1|      Laptop|  Main Warehouse|           10|           40|          NO|
|         2|       Mouse|  Main Warehouse|           20|           30|          NO|
|         3|    Keyboard|Backup Warehouse|           15|           20|          NO|
+----------+------------+----------------+-------------+-------------+------------+

